In [11]:
# import packages and BTMS_model

import os
import sys
import io
import contextlib
import matplotlib.pyplot as plt
import itertools
import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

PROJECT_ROOT = os.path.abspath("../..")
sys.path.append(PROJECT_ROOT)

import lib.BTMS_model as BTMS_model
import CoolProp.CoolProp as CP

In [12]:
# result-file naming configuration
# User-selected filename convention:
# [RESULTID]_[SCN]_[BTMS]_[MODEL]_[TASK].[ext]
# This notebook is for Park liquid-cooling parameter scanning using the 1D model.

CASE_ID = 'C0003'
REVISION = 'R00'
RESULT_ID = f'{CASE_ID}{REVISION}'

SCENARIO_CODE = 'PARK25'
BTMS_CODE = 'LC'
MODEL_CODE = '1D'
TASK_CODE = 'SCAN'
# This scan varies m_dot, T_water_in, and num_cell_seg.
# The task suffix SCAN is intentionally included here to match the selected
# notebook/result naming: C0003R00_PARK25_LC_1D_SCAN.*
RESULT_BASENAME = f'{RESULT_ID}_{SCENARIO_CODE}_{BTMS_CODE}_{MODEL_CODE}_{TASK_CODE}'
NOTEBOOK_NAME = f'{RESULT_BASENAME}.ipynb'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'out', CASE_ID)
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULT_XLSX_NAME = f'{RESULT_BASENAME}.xlsx'
RESULT_XLSX_PATH = os.path.join(OUTPUT_DIR, RESULT_XLSX_NAME)

print(f'Result basename: {RESULT_BASENAME}')
print(f'Notebook file: {NOTEBOOK_NAME}')
print(f'Official Excel result: {RESULT_XLSX_PATH}')


Result basename: C0003R00_PARK25_LC_1D_SCAN
Notebook file: C0003R00_PARK25_LC_1D_SCAN.ipynb
Official Excel result: d:\Workspace\eBATS\out\C0003\C0003R00_PARK25_LC_1D_SCAN.xlsx


In [13]:
# read Qbat(W) from the data file

# Keep the physical simulation duration unchanged, while using the original time step.
SIM_DURATION_S = 1920.0   # s, physical mission duration
dt = 1.0                 # s, integration time step

NUM_STEPS = int(round(SIM_DURATION_S / dt))
if not np.isclose(NUM_STEPS * dt, SIM_DURATION_S):
    raise ValueError('SIM_DURATION_S must be an integer multiple of dt.')

# Keep the original variable name for downstream compatibility.
# Here SIM_TIME_S means the number of numerical time steps, not the physical duration.
SIM_TIME_S = NUM_STEPS

file_name = os.path.join(
    PROJECT_ROOT,
    'data',
    'MD05E070207A1  data_power gen_single cell_Liu_20260421.xlsx'
)

df = pd.read_excel(file_name, sheet_name='Sheet1')
power_generation_data_1s = df['Qbat(W)'].dropna().to_numpy(dtype=float)

# The source Qbat data are treated as 1-s samples. For dt = 1.0 s,
# each 1-s heat-generation value is used for one numerical time step.
time_s = np.arange(SIM_TIME_S + 1) * dt
power_indices = np.floor(time_s[:-1]).astype(int)
power_indices = np.clip(power_indices, 0, len(power_generation_data_1s) - 1)
power_generation_data = power_generation_data_1s[power_indices]

print(f'Simulation duration: {SIM_DURATION_S:.1f} s')
print(f'Time step dt: {dt:.3f} s')
print(f'Number of numerical steps: {SIM_TIME_S}')
print(f'Q_gen array length: {len(power_generation_data)}')

Simulation duration: 1920.0 s
Time step dt: 1.000 s
Number of numerical steps: 1920
Q_gen array length: 1920


In [14]:
# model settings and parameter-scan ranges
#
# Maintenance note:
# In most cases, only edit the dictionaries/lists in this cell.
# The original variable names are kept below so that the downstream
# calculation logic and notebook structure do not need to be changed.

# battery cell properties
# Initial battery temperature is set equal to the scanned inlet water temperature in each case.
BATTERY_PROPS = {
    'm_bat': 48e-3,          # kg
    'cp_bat': 830.0,         # J/(kg K)
    'D_bat': 18e-3,          # m
    'H_bat': 65e-3,          # m
}

# 1D module layout
MODULE_LAYOUT = {
    'N_r': 16,               # number of cells along the coolant-flow direction
    'N_c': 20,               # number of cells in the transverse direction
}

# coolant / water settings
# Note: the liquid-cooling working fluid is water. The air-cooling notebook uses air.
LIQUID_SETTINGS = {
    'fluid': 'Water',
    'p_water': 101325.0,     # Pa
    'D_channel': 4e-3,       # m, circular cooling-channel diameter
    'L_channel': 0.37971337, # m, effective channel length along the module
    'plate_thickness': 0.012,# m, equivalent conduction thickness
    'k_plate': 202.4,        # W/(m K), equivalent plate thermal conductivity
}

# liquid-cooling operating period
# For the liquid-cooling scan, water cooling is kept active during the full mission.
# If the liquid-cooling system needs to follow the same cruise-only logic as the
# air-cooling case, set the corresponding time condition in run_one_case.
COOLING_PERIOD = {
    't_cruise_start': 180.0,
    't_cruise_end': 1560.0,
}

# solver settings passed to BTMS_model.solve_coolant_temperature_distribution
SOLVER_SETTINGS = {
    'solver_tol': 1e-6,
    'solver_maxiter': 1000,
}

# mission-stage definitions for table indicators
mission_stages = [
    ('takeoff',       0.0,    6.0),
    ('climb',         6.0,   36.0),
    ('transition1',  36.0,  180.0),
    ('cruise',      180.0, 1560.0),
    ('transition2',1560.0, 1704.0),
    ('descent',    1704.0, 1734.0),
    ('hover',      1734.0, 1914.0),
    ('landing',    1914.0, 1920.0),
]

# parameter-scan ranges
# Engineering-oriented full-factorial scan:
# - m_dot: 0.5 to 2.0 times the validated baseline flow rate
# - T_water: 20 to 35 degC
# - num_cell_seg: equivalent number of cells represented by each battery segment
M_DOT_BASE = 0.0025445  # kg/s

SCAN_VALUES = {
    'm_dot_list': np.array([
        0.50,
        0.75,
        1.00,
        1.50,
        3.00,   
    ], dtype=float) * M_DOT_BASE,

    'T_water_list': np.array([
        20.0,
        25.0,
        30.0,
        35.0,
    ], dtype=float),

    'num_cell_seg_list': np.array([
        2.0/3.0,     # 30 parallel channels
        2.0 /2.5,    # 25 parallel channels  
        1.00,        # 20 parallel channels
        2.0/1.5,       # 15 parallel channels
        2.00,        # 10 parallel channels
    ], dtype=float),
}

# Keep original variable names for the rest of the notebook.
m_bat = BATTERY_PROPS['m_bat']
cp_bat = BATTERY_PROPS['cp_bat']
D_bat = BATTERY_PROPS['D_bat']
H_bat = BATTERY_PROPS['H_bat']
A_battery = np.pi * D_bat * H_bat

N_r = MODULE_LAYOUT['N_r']
N_c = MODULE_LAYOUT['N_c']
num_seg = N_r

fluid = LIQUID_SETTINGS['fluid']
p_water = LIQUID_SETTINGS['p_water']
D_channel = LIQUID_SETTINGS['D_channel']
L_channel = LIQUID_SETTINGS['L_channel']
plate_thickness = LIQUID_SETTINGS['plate_thickness']
k_plate = LIQUID_SETTINGS['k_plate']

t_cruise_start = COOLING_PERIOD['t_cruise_start']
t_cruise_end = COOLING_PERIOD['t_cruise_end']

solver_tol = SOLVER_SETTINGS['solver_tol']
solver_maxiter = SOLVER_SETTINGS['solver_maxiter']

m_dot_list = SCAN_VALUES['m_dot_list']
T_water_list = SCAN_VALUES['T_water_list']
num_cell_seg_list = SCAN_VALUES['num_cell_seg_list']

In [15]:
# build parameter-scan cases

def build_scan_cases(m_dot_list, T_water_list, num_cell_seg_list):
    """Build all parameter combinations and assign subcase IDs under CASE_ID."""
    cases = []

    for case_count, (m_dot, T_water, num_cell_seg) in enumerate(
        itertools.product(m_dot_list, T_water_list, num_cell_seg_list),
        start=1
    ):
        cases.append({
            'case_id': f'{CASE_ID}_S{case_count:03d}',
            'm_dot': float(m_dot),
            'T_water': float(T_water),
            'num_cell_seg': float(num_cell_seg),
        })

    return cases, pd.DataFrame(cases)


scan_cases, scan_cases_df = build_scan_cases(m_dot_list, T_water_list, num_cell_seg_list)

print(f'Total cases: {len(scan_cases)}')
scan_cases_df.head()

Total cases: 100


,case_id,m_dot,T_water,num_cell_seg
0,C0003_S001,0.001272,20.0,0.666667
1,C0003_S002,0.001272,20.0,0.800000
2,C0003_S003,0.001272,20.0,1.000000
3,C0003_S004,0.001272,20.0,1.333333
4,C0003_S005,0.001272,20.0,2.000000


In [16]:
# calculate water properties once for each inlet water temperature

water_props_cache = {}

for T_water in sorted(scan_cases_df['T_water'].unique()):
    T_water = float(T_water)
    T_water_K = T_water + 273.15

    mu_water = CP.PropsSI('V', 'T', T_water_K, 'P', p_water, fluid)
    rho_water = CP.PropsSI('D', 'T', T_water_K, 'P', p_water, fluid)
    cp_water = CP.PropsSI('C', 'T', T_water_K, 'P', p_water, fluid)
    k_water = CP.PropsSI('L', 'T', T_water_K, 'P', p_water, fluid)
    Pr_water = cp_water * mu_water / k_water

    water_props_cache[T_water] = {
        'mu_water': mu_water,
        'rho_water': rho_water,
        'cp_water': cp_water,
        'k_water': k_water,
        'Pr_water': Pr_water,
    }

print(f'Water properties calculated for {len(water_props_cache)} inlet temperatures.')

Water properties calculated for 4 inlet temperatures.


In [17]:
# run one parameter-scan case by calling BTMS_model

def run_one_case(case_id, m_dot, T_water, num_cell_seg):
    water_props = water_props_cache[float(T_water)]

    # geometry and hydraulic parameters derived from liquid-cooling settings
    A_cool_cs = np.pi * D_channel ** 2 / 4
    A_HT_seg = np.pi * D_channel * L_channel / num_seg

    u_water = m_dot / (water_props['rho_water'] * A_cool_cs)
    Re_water = water_props['rho_water'] * u_water * D_channel / water_props['mu_water']
    Nu_water = BTMS_model.liquid_nusselt_number(Re_water, water_props['Pr_water'], heating=False)
    h_water = Nu_water * water_props['k_water'] / D_channel
    R_plate = plate_thickness / k_plate
    htc_global = 1.0 / (1.0 / h_water + R_plate)
    htc_cool = np.ones(num_seg) * htc_global

    # Initial battery temperature is set equal to the inlet water temperature for each case.
    T_bat = np.ones(num_seg) * T_water
    T_cool = np.ones(num_seg) * T_water

    T_bat_history = np.zeros((SIM_TIME_S + 1, num_seg))
    T_water_history = np.zeros((SIM_TIME_S + 1, num_seg))
    T_bat_history[0, :] = T_bat
    T_water_history[0, :] = T_cool

    for k in range(SIM_TIME_S):
        t = time_s[k]
        is_cool = True

        # Liquid-cooled 1D model: one battery node corresponds to one coolant-domain node.
        # num_cell_seg is passed here as the equivalent number of cells represented by each battery segment.
        args = {
            'num_seg': num_seg,
            'num_seg_bat': num_seg,
            'dt': dt,
            'T_cool_pre': T_cool,
            'T_bat_pre': T_bat,
            'u_cool_in': u_water,
            'p_cool': p_water,
            'fluid_cool': fluid,
            'A_HT_seg': A_HT_seg,
            'A_cool_cs': A_cool_cs,
            'm_bat': m_bat,
            'cp_bat': cp_bat,
            'D_bat': D_bat,
            'T_cool_in': T_water,
            'T_cool_out': max(float(T_cool[-1]), T_water),
            'is_cool': is_cool,
            'htc_cool': htc_cool,
            'cp_cool': water_props['cp_water'],
            'rho_cool': water_props['rho_water'],
            'Q_gen': power_generation_data[k],
            'num_cell_seg': num_cell_seg,
            'debug': False,
        }

        try:
            with contextlib.redirect_stdout(io.StringIO()):
                T_dist = BTMS_model.solve_coolant_temperature_distribution(
                    args,
                    tol=solver_tol,
                    maxiter=solver_maxiter,
                    debug=False
                )
        except Exception as e:
            print("\nSolver failed inside run_one_case.")
            print(f"case_id = {case_id}")
            print(f"m_dot = {m_dot} kg/s")
            print(f"T_water_in = {T_water} degC")
            print(f"num_cell_seg = {num_cell_seg}")
            print(f"time step k = {k}")
            print(f"t = {t:.1f} s")
            print(f"is_cool = {is_cool}")
            print(f"Q_gen = {power_generation_data[k]} W")
            print(f"u_water = {u_water}")
            print(f"Re_water = {Re_water}")
            print(f"Nu_water = {Nu_water}")
            print(f"h_water = {h_water}")
            print(f"htc_global = {htc_global}")
            print(f"A_HT_seg = {A_HT_seg}")
            print(f"A_cool_cs = {A_cool_cs}")
            print(f"T_bat_min/max before solve = {np.min(T_bat)}, {np.max(T_bat)}")
            print(f"T_cool_min/max before solve = {np.min(T_cool)}, {np.max(T_cool)}")
            print(f"solver_tol = {solver_tol}")
            print(f"solver_maxiter = {solver_maxiter}")
            print(f"error = {repr(e)}")
            raise

        if not np.all(np.isfinite(T_dist)):
            print("\nSolver returned non-finite values.")
            print(f"case_id = {case_id}")
            print(f"m_dot = {m_dot} kg/s")
            print(f"T_water_in = {T_water} degC")
            print(f"num_cell_seg = {num_cell_seg}")
            print(f"time step k = {k}")
            print(f"t = {t:.1f} s")
            print(f"T_dist_min/max = {np.nanmin(T_dist)}, {np.nanmax(T_dist)}")
            raise FloatingPointError("Non-finite values detected in T_dist")

        T_cool = T_dist[:num_seg]
        T_bat = T_dist[num_seg:]

        T_bat_history[k + 1, :] = T_bat
        T_water_history[k + 1, :] = T_cool

    Tmax = np.max(T_bat_history, axis=1)
    Tmin = np.min(T_bat_history, axis=1)
    DeltaT = Tmax - Tmin
    Twater_out = T_water_history[:, -1]

    def idx(t):
        return int(round(t / dt))

    row = {
        'case_id': case_id,
        'm_dot_kg_s': m_dot,
        'T_water_in_C': T_water,
        'num_cell_seg': num_cell_seg,
        'Tmax_mission_C': float(np.max(Tmax)),
        'DeltaT_mission_max_C': float(np.max(DeltaT)),
        'cruise_recovery_Tmax_C': float(Tmax[idx(t_cruise_start)] - Tmax[idx(t_cruise_end)]),
        'Twater_out_cruise_end_C': float(Twater_out[idx(t_cruise_end)]),
    }

    for stage_name, t0, t1 in mission_stages:
        row[f'dTmax_{stage_name}_C'] = float(Tmax[idx(t1)] - Tmax[idx(t0)])

    return row

In [ ]:
# run all cases and export the final xlsx table

def format_excel_table(xlsx_path):
    """Apply the original Times New Roman table style to the exported Excel file."""
    wb = load_workbook(xlsx_path)
    ws = wb.active
    ws.title = 'Parameter scan'

    body_font = Font(name='Times New Roman', size=10)
    header_font = Font(name='Times New Roman', size=10, bold=True)
    alignment = Alignment(horizontal='center', vertical='center')
    thin = Side(style='thin')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for row in ws.iter_rows():
        for cell in row:
            cell.font = header_font if cell.row == 1 else body_font
            cell.alignment = alignment
            cell.border = border
            if cell.row > 1 and isinstance(cell.value, float):
                cell.number_format = '0.0000'

    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions

    for col_idx, column_cells in enumerate(ws.columns, start=1):
        max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 12), 28)

    wb.save(xlsx_path)

summary_rows = []

for n, case in enumerate(scan_cases, start=1):
    print(
        f"Running {n}/{len(scan_cases)}: {case['case_id']}, "
        f"m_dot={case['m_dot']}, "
        f"T_water={case['T_water']}, "
        f"num_cell_seg={case['num_cell_seg']}"
    )

    try:
        summary_rows.append(run_one_case(**case))
    except Exception as e:
        print("\nParameter scan stopped because one case failed.")
        print(f"failed index = {n}/{len(scan_cases)}")
        print(f"failed case_id = {case['case_id']}")
        print(f"failed m_dot = {case['m_dot']} kg/s")
        print(f"failed T_water_in = {case['T_water']} degC")
        print(f"failed num_cell_seg = {case['num_cell_seg']}")
        print(f"error = {repr(e)}")
        raise

parameter_scan_table = pd.DataFrame(summary_rows)

column_order = [
    'case_id',
    'm_dot_kg_s',
    'T_water_in_C',
    'num_cell_seg',
    'Tmax_mission_C',
    'DeltaT_mission_max_C',
    'cruise_recovery_Tmax_C',
    'Twater_out_cruise_end_C',
    'dTmax_takeoff_C',
    'dTmax_climb_C',   
    'dTmax_transition1_C',
    'dTmax_cruise_C',
    'dTmax_transition2_C',
    'dTmax_descent_C',
    'dTmax_hover_C',
    'dTmax_landing_C',
]

parameter_scan_table = parameter_scan_table[column_order]

output_dir = OUTPUT_DIR
os.makedirs(output_dir, exist_ok=True)

xlsx_path = os.path.join(output_dir, RESULT_XLSX_NAME)

parameter_scan_table.to_excel(xlsx_path, index=False)
format_excel_table(xlsx_path)

print(f'Saved official Excel: {xlsx_path}')
print('Only one table file is generated for this case.')

parameter_scan_table.head()


SyntaxError: invalid syntax (4108607635.py, line 3)